In [37]:
import sys
import os

# Add the src/ folder to Python's path so we can import from it
sys.path.insert(0, os.path.abspath("../src"))

print("Path configured. Ready to import.")


Path configured. Ready to import.


In [38]:
from ingestion import load_and_chunk_papers
from retrieval import get_or_build_vector_store, query, print_results
from rag_chain import build_rag_chain, ask, print_answer

print("Imports successful.")

Imports successful.


In [39]:
# This walks data/papers/, loads every PDF, and splits into chunks
chunks = load_and_chunk_papers(papers_dir="../data/papers")

print(f"\nTotal chunks created: {len(chunks)}")
print(f"Expected range: 1000–6000 for 20–30 papers")

Found 23 PDF(s). Loading...
  Processing: A survey of deep learning methods and datasets for hand pose.pdf
  Processing: A_Review_on_3D_Hand_Pose_and_Shape_Reconstruction_from_Color_Images.pdf
  Processing: Advances in vision-based deep learning methods for interacting hands reconstruction A survey.pdf
  Processing: Best-Known-Performance-Optimizations-for-D400-Stereo-Cameras-over-Lifetime-rev-1.31.pdf
  Processing: Doosti_HOPE-Net_A_Graph-Based_Model_for_Hand-Object_Pose_Estimation_CVPR_2020_paper.pdf
  Processing: Hasson_Learning_Joint_Reconstruction_of_Hands_and_Manipulated_Objects_CVPR_2019_paper.pdf
  Processing: Jian_AffordPose_A_Large-Scale_Dataset_of_Hand-Object_Interactions_with_Affordance-Driven_Hand_ICCV_2023_paper.pdf
  Processing: Kwon_H2O_Two_Hands_Manipulating_Objects_for_First_Person_Interaction_Recognition_ICCV_2021_paper.pdf
  Processing: Liu_HOI4D_A_4D_Egocentric_Dataset_for_Category-Level_Human-Object_Interaction_CVPR_2022_paper.pdf
  Processing: Malik_HandVoxNet_De

In [40]:
# Always inspect a sample before embedding — catch bad PDFs early
sample = chunks[0]

print("=== Sample Chunk ===")
print(f"Source   : {sample.metadata['source']}")
print(f"Page     : {sample.metadata['page']}")
print(f"Char len : {len(sample.page_content)}")
print(f"\nContent preview:\n{sample.page_content[:500]}")

=== Sample Chunk ===
Source   : A survey of deep learning methods and datasets for hand pose.pdf
Page     : 0
Char len : 964

Content preview:
Computers & Graphics 116 (2023) 474–490
Contents lists available at ScienceDirect
Computers & Graphics
journal homepage: www.elsevier.com/locate/cag
Survey Paper
A survey of deep learning methods and datasets for hand pose
estimation from hand-object interaction images✩
Taeyun Woo, Wonjung Park, Woohyun Jeong, Jinah Park ∗
Korea Advanced Institution of Science and Technology, Daejeon, Republic of Korea
a r t i c l e i n f o
Article history:
Received 12 April 2023
Received in revised form 28 July


In [41]:
from collections import Counter

sources = [chunk.metadata["source"] for chunk in chunks]
counts = Counter(sources)

print(f"{'Paper':<45} {'Chunks':>6}")
print("-" * 53)
for paper, count in sorted(counts.items()):
    print(f"{paper:<45} {count:>6}")

Paper                                         Chunks
-----------------------------------------------------
A survey of deep learning methods and datasets for hand pose.pdf    162
A_Review_on_3D_Hand_Pose_and_Shape_Reconstruction_from_Color_Images.pdf     45
Advances in vision-based deep learning methods for interacting hands reconstruction A survey.pdf    114
Best-Known-Performance-Optimizations-for-D400-Stereo-Cameras-over-Lifetime-rev-1.31.pdf     38
Doosti_HOPE-Net_A_Graph-Based_Model_for_Hand-Object_Pose_Estimation_CVPR_2020_paper.pdf     53
Hasson_Learning_Joint_Reconstruction_of_Hands_and_Manipulated_Objects_CVPR_2019_paper.pdf     66
Jian_AffordPose_A_Large-Scale_Dataset_of_Hand-Object_Interactions_with_Affordance-Driven_Hand_ICCV_2023_paper.pdf     68
Kwon_H2O_Two_Hands_Manipulating_Objects_for_First_Person_Interaction_Recognition_ICCV_2021_paper.pdf     75
Liu_HOI4D_A_4D_Egocentric_Dataset_for_Category-Level_Human-Object_Interaction_CVPR_2022_paper.pdf     64
Malik_HandVoxNet_

In [42]:
# First run: embeds everything and saves to disk (~5-15 min depending on your machine)
# Subsequent runs: loads from disk in seconds
vector_store = get_or_build_vector_store(papers_dir="../data/papers")

Existing vector store found (1591 chunks). Loading...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5958.16it/s]


Loaded vector store: 1591 chunks in 'cv_papers'


In [43]:
count = vector_store._collection.count()
print(f"Chunks stored in ChromaDB: {count}")
print(f"Matches chunks created   : {len(chunks)}")
print(f"All accounted for        : {count == len(chunks)}")

Chunks stored in ChromaDB: 1591
Matches chunks created   : 1591
All accounted for        : True


In [44]:
# Build the RAG chain
chain, retriever = build_rag_chain(vector_store)  

test_questions = [
    "What is the mAP of Fast YOLO on PASCAL VOC 2007?",
    "What dataset does HOPE-Net use for evaluation?",
    "How does HandOccNet handle occlusion?",
    "What is the main contribution of HOI4D dataset?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    result = ask(chain, retriever, q)  
    print_answer(result)


Q: What is the mAP of Fast YOLO on PASCAL VOC 2007?


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktk8hf58fz6bsrq99nka7jwv` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99042, Requested 1787. Please try again in 11m56.256s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
single_question = "What tracking method does PhysTwin use?"
result = ask(chain, retriever, single_question)
print_answer(result)


ANSWER:
+1… … 
Stiffness
Simulated Geometry and MotionGaussian Rendering
Figure 2. Overview of Our PhysTwin Framework. We present an overview of our PhysTwin framework, where the core representation
includes geometry, topology, physical parameters (associated with springs and contacts), and Gaussian kernels. To optimize PhysTwin,
we minimize the rendering loss and the discrepancy between simulated and observed geometry/motion. The rendering loss optimizes the

What is the main goal of the PhysTwin framework, and what are the two stages of optimization in this framework?
The main goal of the PhysTwin framework is to optimize the core representation, which includes geometry, topology, physical parameters, and Gaussian kernels, by minimizing the rendering loss and the discrepancy between simulated and observed geometry/motion (phystwin.pdf, Page: 2). 
The two stages of optimization in the PhysTwin framework are: 
1. The first stage focuses on optimizing the geometry and physical paramete

In [ ]:
evaluation = [
    {
        "query": "What is the mAP of Fast YOLO on PASCAL VOC 2007?",
        "expected_paper": "Yolo.pdf",
        "top_result_source": None,
        "correct": None,
    },
    {
        "query": "What tracking method does PhysTwin use?",
        "expected_paper": "phystwin.pdf",
        "top_result_source": None,
        "correct": None,
    },
]

# Run each query and fill in top result automatically
for e in evaluation:
    result = ask(chain, retriever, e["query"])
    e["top_result_source"] = result["sources"][0]["source"] if result["sources"] else "None"

# Print scorecard — manually set "correct" to True/False after inspecting
for e in evaluation:
    print(f"Query    : {e['query'][:55]}")
    print(f"Expected : {e['expected_paper']}")
    print(f"Got      : {e['top_result_source']}")
    print()

Query    : What is the mAP of Fast YOLO on PASCAL VOC 2007?
Expected : Yolo.pdf
Got      : Yolo.pdf

Query    : What tracking method does PhysTwin use?
Expected : phystwin.pdf
Got      : phystwin.pdf



In [ ]:
evaluation = [
    {"query": "What is the mAP of Fast YOLO on PASCAL VOC 2007?", "expected": "Yolo.pdf"},
    {"query": "HOPE-Net affordance graph hand object pose estimation evaluation","expected": "Doosti_HOPE-Net"},    
    {"query": "What is the main contribution of HOI4D dataset?", "expected": "Liu_HOI4D...pdf"},
    {"query": "HandOccNet occlusion-robust 3D hand mesh estimation network architecture", "expected": "Park_HandOccNet"},
    {"query": "is ObMan containing deformable objects as well as rigid objects", "expected": "Hasson"},
    {"query": "What is PhysTwin's core representation?", "expected": "phystwin.pdf"},
    {"query": "What bandwidth is needed for four RealSense depth cameras?", "expected": "Best-Known"},
    {"query": "What is the main task in AffordPose dataset?", "expected": "Jian_AffordPose...pdf"},
    {"query": "How does HOIDiffusion generate interaction data?", "expected": "Zhang_HOIDiffusion...pdf"},
    {"query": "What method does SHOWMe benchmark evaluate?", "expected": "Swamy_SHOWMe...pdf"},
]

correct = 0
for e in evaluation:
    result = ask(chain, retriever, e["query"])
    top_source = result["sources"][0]["source"] if result["sources"] else "None"
    expected = e["expected"]
    is_correct = expected.split("_")[0].lower() in top_source.lower()
    if is_correct:
        correct += 1
    status = "✓" if is_correct else "✗"
    print(f"{status} Q: {e['query'][:50]}")
    print(f"  Expected : {expected[:50]}")
    print(f"  Got      : {top_source[:50]}")
    print()

print(f"\nFinal score: {correct}/10")

✓ Q: What is the mAP of Fast YOLO on PASCAL VOC 2007?
  Expected : Yolo.pdf
  Got      : Yolo.pdf



RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktk8hf58fz6bsrq99nka7jwv` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99414, Requested 1837. Please try again in 18m0.864s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
edge_cases = [
    "What is the best pizza recipe?",
    "Who is the president of the United States?",
    "How do I invest in stocks?",
]

for q in edge_cases:
    print(f"Q: {q}")
    result = ask(chain, retriever, q)
    print(f"A: {result['answer'][:200]}")
    print()

Q: What is the best pizza recipe?


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktk8hf58fz6bsrq99nka7jwv` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 97733, Requested 2873. Please try again in 8m43.584s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}